# Required Tasks Tri-Engine Notebook

这个 notebook 改为“每个任务一个配置文件”的工作范式：配置文件直接声明 QASM、case、引擎和展示指标，然后逐条调用现有 `run_workflow` 执行。

结果目录固定写到 `examples/noise_simulation_tests/runs/required_tasks_tri_engine/`。

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve().parent.parent
if not (ROOT / 'pyproject.toml').exists() or not (ROOT / 'src').exists():
    raise RuntimeError('Please run this notebook with the repository root as the current working directory.')

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.analysis.trace_semantics import extract_p1_series
from qsim.ui.notebook import run_workflow
from qsim.ui.result_summary import attach_compare_status, summarize_workflow_result

BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
TASK_CONFIG_DIR = ROOT / 'examples' / 'noise_simulation_tests' / 'required_tasks_tri_engine'
OUT_ROOT = ROOT / 'examples' / 'noise_simulation_tests' / 'runs' / 'required_tasks_tri_engine'
SUMMARY_ROOT = OUT_ROOT / '_summaries'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

TASK_CONFIGS = [
    json.loads(path.read_text(encoding='utf-8'))
    for path in sorted(TASK_CONFIG_DIR.glob('*.json'))
]

print('ROOT =', ROOT)
print('TASK_CONFIG_DIR =', TASK_CONFIG_DIR)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)
print('task_count =', len(TASK_CONFIGS))


ROOT = D:\超导量子计算机噪声抑制\qsim
TASK_CONFIG_DIR = D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\required_tasks_tri_engine
BACKEND_PATH = D:\超导量子计算机噪声抑制\qsim\examples\backend.yaml
OUT_ROOT = D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine
task_count = 7


In [3]:
task_overview = pd.DataFrame(
    [
        {
            'task': task['tag'],
            'title': task['title'],
            'num_cases': len(task['cases']),
            'engines': ', '.join(task['engines']),
            'solver_mode': task.get('solver_mode', 'me'),
        }
        for task in TASK_CONFIGS[0:1]
    ]
)
display(task_overview)


,task,title,num_cases,engines,solver_mode
0,task1_single_qubit_baseline,Task 1 单比特基准模型,2,"qutip, julia_qtoolbox, julia_qoptics",me


In [4]:
SCALAR_SEMANTIC_METRICS = {'final_p1_obs', 'final_p0_obs', 'mean_excited_obs'}


def _metric_usable_rows(df: pd.DataFrame, metric: str) -> tuple[pd.DataFrame, str]:
    if metric in SCALAR_SEMANTIC_METRICS:
        usable = df[df[metric].notna()].copy()
        return usable, 'semantic-scalar-comparable'
    usable = df[df['compare_status'] == 'pointwise-comparable'].copy()
    return usable, 'pointwise-comparable'


def plot_metric(df: pd.DataFrame, metric: str, title: str, ylabel: str | None = None):
    usable, mode = _metric_usable_rows(df, metric)
    if usable.empty:
        print(f'{title}: skipped plot because no rows are usable under mode={mode}.')
        return
    pivot = usable.pivot(index='case', columns='engine', values=metric)
    pivot = pivot.dropna(how='all')
    if pivot.empty:
        print(f'{title}: skipped plot because pivot is empty under mode={mode}.')
        return
    ax = pivot.plot(kind='bar', figsize=(9, 4), rot=0)
    ax.set_title(title)
    ax.set_ylabel(ylabel or metric)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_p1_dynamics(dyn_df: pd.DataFrame, task_title: str):
    if dyn_df is None or dyn_df.empty:
        print(f'{task_title}: p1(t) dynamics skipped because no dynamics rows were collected.')
        return
    for case in sorted(dyn_df['case'].unique()):
        case_df = dyn_df[dyn_df['case'] == case].sort_values('engine')
        if case_df.empty:
            continue
        fig, ax = plt.subplots(figsize=(9, 4))
        for _, row in case_df.iterrows():
            ax.plot(row['times'], row['p1_t'], label=row['engine'])
        ax.set_title(f'{task_title}: {case} p1(t) dynamics')
        ax.set_xlabel('time')
        ax.set_ylabel('p1')
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()


def display_task(df: pd.DataFrame, task_title: str, metrics: list[str], dyn_df: pd.DataFrame | None = None):
    cols = [
        'task', 'case', 'engine', 'state_encoding', 'compare_status', 'compare_reason',
        'state_len', 'final_state_json', 'final_state_sum', 'final_p0_obs', 'final_p1_obs', 'mean_excited_obs',
    ]
    pulse_cols = [c for c in ['RO_0_duration', 'RO_0_abs_area', 'RO_0_peak', 'XY_0_abs_area'] if c in df.columns]
    extra_cols = [c for c in ['solver_impl', 'native_solver', 'note'] if c in df.columns]
    show = [c for c in cols + pulse_cols + extra_cols if c in df.columns]
    print(task_title)
    display(df[show].sort_values(['case', 'engine']).reset_index(drop=True))
    pointwise_cases = sorted(df.loc[df['compare_status'] == 'pointwise-comparable', 'case'].unique())
    scalar_cases = sorted(df.loc[df['final_p1_obs'].notna(), 'case'].unique()) if 'final_p1_obs' in df.columns else []
    print('Pointwise-comparable cases =', pointwise_cases)
    print('Semantic-scalar-comparable cases (final_p1_obs) =', scalar_cases)
    for metric in metrics:
        if metric in df.columns:
            plot_metric(df, metric, f'{task_title}: {metric}')
    plot_p1_dynamics(dyn_df, task_title)


## 执行

下面这一格直接读取任务配置目录，逐条调用 `run_workflow`。notebook 只保留展示逻辑，结果汇总由 `src/qsim/ui/result_summary.py` 提供。

In [5]:
rows = []
task_frames: dict[str, pd.DataFrame] = {}
task_dyn_frames: dict[str, pd.DataFrame] = {}

for task in TASK_CONFIGS[0:1]:
    task_rows = []
    task_dyn_rows = []
    qasm_text = task['qasm_text']
    solver_mode = task.get('solver_mode', 'me')
    for case in task['cases']:
        hardware = dict(case.get('hardware', {}) or {})
        noise = dict(case.get('noise', {}) or {})
        for engine in task['engines']:
            out_dir = OUT_ROOT / task['tag'] / case['tag'] / engine
            result = run_workflow(
                qasm_text=qasm_text,
                backend_path=str(BACKEND_PATH),
                out_dir=str(out_dir),
                hardware=hardware,
                noise=noise,
                engine=engine,
                solver_mode=solver_mode,
                persist_artifacts=True,
                export_dxf=False,
                export_plots=False,
                decoder='mock',
                allow_mock_fallback=False,
            )
            row = summarize_workflow_result(
                result,
                task_tag=task['tag'],
                task_title=task['title'],
                case_tag=case['tag'],
                engine=engine,
                hardware=hardware,
                noise=noise,
                note=case.get('note', ''),
            )
            task_rows.append(row)
            rows.append(row)
            p1_t = extract_p1_series(result['trace'])
            times = [float(x) for x in result['trace'].times[: len(p1_t)]]
            task_dyn_rows.append({'task': task['tag'], 'case': case['tag'], 'engine': engine, 'times': times, 'p1_t': p1_t})
    task_df = attach_compare_status(pd.DataFrame(task_rows))
    csv_path = SUMMARY_ROOT / f"{task['tag']}_summary.csv"
    task_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"[{task['tag']}] wrote", csv_path)
    task_frames[task['tag']] = task_df
    task_dyn_frames[task['tag']] = pd.DataFrame(task_dyn_rows)
    display_task(task_df, task['title'], task.get('display_metrics', []), dyn_df=task_dyn_frames[task['tag']])

all_df = attach_compare_status(pd.DataFrame(rows))
all_summary_csv = SUMMARY_ROOT / 'all_tasks_summary.csv'
all_df.to_csv(all_summary_csv, index=False, encoding='utf-8-sig')
print('all summary ->', all_summary_csv)


[task1_single_qubit_baseline] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task1_single_qubit_baseline_summary.csv
Task 1 单比特基准模型


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task1_single_qubit_baseline,baseline,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.3421061546928099, 0.6578938453071901]",1.000000,0.342106,0.657894,0.665048,240.0,158.4,0.8,16.400267,quantumoptics.timeevolution.master,True,baseline single-qubit model
1,task1_single_qubit_baseline,baseline,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.3421085300810429, 0.6578914699189571]",1.000000,0.342109,0.657891,0.626677,240.0,158.4,0.8,16.400267,quantumtoolbox.mesolve,True,baseline single-qubit model
2,task1_single_qubit_baseline,baseline,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.9997596199783175],0.999760,0.000240,0.999760,0.879648,240.0,158.4,0.8,16.400267,,False,baseline single-qubit model
3,task1_single_qubit_baseline,detuned,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.35332307589571843, 0.6466769241042816]",1.000000,0.353323,0.646677,0.651699,256.0,158.4,0.8,22.962654,quantumoptics.timeevolution.master,True,detuned and slower gate
4,task1_single_qubit_baseline,detuned,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.35333127697369227, 0.6466687230263077]",1.000000,0.353331,0.646669,0.627742,256.0,158.4,0.8,22.962654,quantumtoolbox.mesolve,True,detuned and slower gate
5,task1_single_qubit_baseline,detuned,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.9999962818466597],0.999996,0.000004,0.999996,0.919580,256.0,158.4,0.8,22.962654,,False,detuned and slower gate


Pointwise-comparable cases = []
Task 1 单比特基准模型: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.
Task 1 单比特基准模型: final_state_last: skipped plot because no case is pointwise-comparable across engines.
all summary -> D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\all_tasks_summary.csv


## 汇总

把 7 个任务的摘要表合并，并做一个面向任务、引擎和语义状态的整体统计。

In [ ]:
summary = (
    all_df.groupby(['task', 'engine', 'state_encoding', 'compare_status'])[['state_len', 'final_state_sum', 'final_p1_obs', 'mean_excited_obs']]
    .agg(['mean', 'min', 'max'])
)
display(summary)
all_df.head()


## 结论检查点

- 路径初始化缩减到最小前提：直接假设 notebook 从 repo root 运行。
- QASM 已进入每个 task 的 JSON 配置文件，不再由 notebook 单独维护。
- `pulse_metrics` / `summarize_result` 类结果整理逻辑已上收到 `src/qsim/ui/result_summary.py`。
- notebook 文本已重新以 UTF-8 落盘。